In [1]:
# =============================================================================
# BLOCO 1 — CONFIGURAÇÃO E EXTRAÇÃO DE DELTAS ACÚSTICOS
# =============================================================================
import os
import math
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from scipy.signal import resample_poly
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

print("Fase 3 Otimizada: Validação de Rigor Científico do Dataset")
print("=" * 80)

# MUDAR ESTAS 3 LINHAS DE DIRETÓRIOS:
FILE_METADATA = "dataset_fase2_valid_elite/metadata.csv"     # <--- Aponta para o CSV novo de elite
DIR_AUDIO_WET = "dataset_fase2_valid_elite/audio"            # <--- Aponta para a pasta nova de áudios modificados
DIR_AUDIO_DRY = "nsynth-valid/audio"                         # <--- CRÍTICO: Mudar de 'test' para 'valid' para achar os áudios secos correspondentes

df_meta = pd.read_csv(FILE_METADATA)
dados_features = []
cache_dry = {}

print("Extraindo Deltas Acústicos (Isolando a matemática do efeito)...")

for index, row in tqdm(df_meta.iterrows(), total=len(df_meta)):
    nome_wet = row['file_name']
    nota_dry = row['original_note']
    
    caminho_wet = os.path.join(DIR_AUDIO_WET, nome_wet)
    caminho_dry = os.path.join(DIR_AUDIO_DRY, f"{nota_dry}.wav")
    
    try:
        # Lemos os dois áudios com Soundfile nativo
        audio_wet, sr = sf.read(caminho_wet)
        
        # Cache e upsampling de alta fidelidade para o áudio original
        if nota_dry not in cache_dry:
            audio_dry, sr_dry = sf.read(caminho_dry)
            if sr_dry != sr:
                fator = math.gcd(sr, sr_dry)
                audio_dry = resample_poly(audio_dry, sr // fator, sr_dry // fator)
            cache_dry[nota_dry] = audio_dry
            
        audio_dry = cache_dry[nota_dry]
        
        # Alinha os tamanhos para garantir consistência vetorial
        tamanho_minimo = min(len(audio_wet), len(audio_dry))
        a_wet = audio_wet[:tamanho_minimo]
        a_dry = audio_dry[:tamanho_minimo]
        
        # Cálculo dos Deltas (Processado - Original)
        mfcc_wet = np.mean(librosa.feature.mfcc(y=a_wet, sr=sr, n_mfcc=20), axis=1)
        mfcc_dry = np.mean(librosa.feature.mfcc(y=a_dry, sr=sr, n_mfcc=20), axis=1)
        delta_mfcc = mfcc_wet - mfcc_dry
        
        cent_wet = np.mean(librosa.feature.spectral_centroid(y=a_wet, sr=sr))
        cent_dry = np.mean(librosa.feature.spectral_centroid(y=a_dry, sr=sr))
        delta_centroid = cent_wet - cent_dry
        
        roll_wet = np.mean(librosa.feature.spectral_rolloff(y=a_wet, sr=sr))
        roll_dry = np.mean(librosa.feature.spectral_rolloff(y=a_dry, sr=sr))
        delta_rolloff = roll_wet - roll_dry

        flat_wet = np.mean(librosa.feature.spectral_flatness(y=a_wet))
        flat_dry = np.mean(librosa.feature.spectral_flatness(y=a_dry))
        delta_flatness = flat_wet - flat_dry

        dados_features.append({
            'file_name': nome_wet,
            'keyword': row['keyword'],
            'intensity': row['intensity'],
            'original_note': nota_dry,
            'mfccs': delta_mfcc,       
            'centroid': delta_centroid, 
            'rolloff': delta_rolloff,
            'flatness': delta_flatness
        })
        
    except Exception as e:
        continue

df_feat = pd.DataFrame(dados_features)
print("\nExtração de Deltas concluída com sucesso.")

Fase 3 Otimizada: Validação de Rigor Científico do Dataset
Extraindo Deltas Acústicos (Isolando a matemática do efeito)...


100%|██████████████████████████████████████████████████████████████████████████████| 9600/9600 [10:56<00:00, 14.62it/s]


Extração de Deltas concluída com sucesso.


In [2]:
# =============================================================================
# BLOCO 2 — MONOTONICIDADE (TAU DE KENDALL) COM DELTAS
# =============================================================================
from scipy.stats import kendalltau

print("\n" + "=" * 80)
print("1. TESTE DE MONOTONICIDADE (TAU DE KENDALL)")
print("-" * 80)

# Mapeamento físico-perceptual refinado para a avaliação dos Deltas
feature_alvo = {
    'bright': ('rolloff', 1),   # Sobe
    'dark':   ('rolloff', -1),  # Desce
    'crisp':  ('flatness', 1),  # Mais harmónicos/ruído
    'muffled':('flatness', -1), # Mais tonal/abafado
    'warm':   ('centroid', -1), # Foco em médios/graves
    'tinny':  ('centroid', 1),  # Foco em agudos metálicos
}

for kw, (feat, weight) in feature_alvo.items():
    df_kw = df_feat[df_feat['keyword'] == kw]
    if df_kw.empty: continue
    
    taus = []
    for nota in df_kw['original_note'].unique():
        df_n = df_kw[df_kw['original_note'] == nota].sort_values('intensity')
        if len(df_n) < 3: continue
        
        t, _ = kendalltau(df_n['intensity'], df_n[feat])
        if not np.isnan(t): taus.append(t * weight)
        
    if taus:
        print(f"  [{kw.upper():10s}] Tau: {np.mean(taus):.3f} (Ideal > 0.7)")


1. TESTE DE MONOTONICIDADE (TAU DE KENDALL)
--------------------------------------------------------------------------------
  [BRIGHT    ] Tau: -0.217 (Ideal > 0.7)
  [DARK      ] Tau: 0.234 (Ideal > 0.7)
  [CRISP     ] Tau: 0.160 (Ideal > 0.7)
  [MUFFLED   ] Tau: 0.755 (Ideal > 0.7)
  [WARM      ] Tau: 0.354 (Ideal > 0.7)
  [TINNY     ] Tau: -0.388 (Ideal > 0.7)


In [3]:
# =============================================================================
# BLOCO 4 — BASELINE MLP (PROVA-DE-CONCEITO)
# =============================================================================
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print("\n" + "=" * 80)
print("3. CLASSIFICADOR BASELINE (PROVA-DE-CONCEITO)")
print("-" * 80)

# Filtramos apenas a intensidade máxima para ver se o modelo distingue os efeitos puros
df_max = df_feat[df_feat['intensity'] == 1.0]

if not df_max.empty:
    X = scaler.fit_transform(np.stack(df_max['mfccs'].values))
    y = df_max['keyword'].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    clf = MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=42).fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))

    print(f"  Acurácia Final do Classificador: {acc*100:.2f}%")
    print(f"  (Chute Aleatório para {len(np.unique(y))} classes: {100/len(np.unique(y)):.2f}%)")
    print("\nValidação Concluída. Se a Acurácia for maior que o acaso e o Tau alto, o dataset está validado.")
else:
    print("Erro: Nenhum dado de intensidade 1.0 encontrado para treinar o classificador.")


3. CLASSIFICADOR BASELINE (PROVA-DE-CONCEITO)
--------------------------------------------------------------------------------


NameError: name 'scaler' is not defined

In [4]:
# =============================================================================
# BLOCO 3 — MMD PADRONIZADO (DISTÂNCIA ENTRE OPOSTOS)
# =============================================================================
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import rbf_kernel

print("\n" + "=" * 80)
print("2. MMD PADRONIZADO (DISTÂNCIA ENTRE OPOSTOS)")
print("-" * 80)

scaler = StandardScaler()
X_all_mfcc = scaler.fit_transform(np.stack(df_feat['mfccs'].values))

def mmd_rbf(X, Y, gamma=0.1):
    XX = rbf_kernel(X, X, gamma)
    YY = rbf_kernel(Y, Y, gamma)
    XY = rbf_kernel(X, Y, gamma)
    return XX.mean() + YY.mean() - 2 * XY.mean()

pares = [("bright", "dark"), ("warm", "tinny"), ("crisp", "muffled")]
for p1, p2 in pares:
    idx1 = df_feat[(df_feat['keyword'] == p1) & (df_feat['intensity'] == 1.0)].index
    idx2 = df_feat[(df_feat['keyword'] == p2) & (df_feat['intensity'] == 1.0)].index
    if len(idx1) > 0 and len(idx2) > 0:
        dist = mmd_rbf(X_all_mfcc[idx1], X_all_mfcc[idx2])
        print(f"  [{p1} vs {p2}] MMD: {dist:.4f} (Ideal > 0.1)")


2. MMD PADRONIZADO (DISTÂNCIA ENTRE OPOSTOS)
--------------------------------------------------------------------------------
  [bright vs dark] MMD: 0.2138 (Ideal > 0.1)
  [warm vs tinny] MMD: 0.0044 (Ideal > 0.1)
  [crisp vs muffled] MMD: 0.2344 (Ideal > 0.1)


In [5]:
import os
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

print("Iniciando Validação com CNN (Espectrogramas)")
print("=" * 80)

# =============================================================================
# 1. CONFIGURAÇÕES E CARREGAMENTO
# =============================================================================
FILE_METADATA = "dataset_fase2_final/metadata.csv"
DIR_AUDIO_WET = "dataset_fase2_final/audio"

df_meta = pd.read_csv(FILE_METADATA)

# Pega apenas amostras de intensidade máxima (como fizemos no MLP)
df_max = df_meta[df_meta['intensity'] == 1.0].copy()

# =============================================================================
# 2. EXTRAÇÃO DE IMAGENS ACÚSTICAS (MEL-ESPECTROGRAMAS)
# =============================================================================
# A CNN precisa de "imagens" do mesmo tamanho. Vamos fixar o tempo (ex: 2 segundos).
MAX_PAD_LEN = 86  # Aproximadamente 2 segundos em 22050Hz (pode ajustar)
N_MELS = 64       # Número de bandas de frequência ("altura" da imagem)

def extrair_espectrograma(audio_path):
    try:
        y, sr = librosa.load(audio_path, sr=22050, mono=True)
        # Gera o Mel-Espectrograma
        melspec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, n_fft=1024, hop_length=512)
        # Converte para decibéis (log scale) - Essencial para a CNN "enxergar" direito
        melspec_db = librosa.power_to_db(melspec, ref=np.max)
        
        # Padroniza a largura da imagem (pad ou truncate)
        if melspec_db.shape[1] > MAX_PAD_LEN:
            melspec_db = melspec_db[:, :MAX_PAD_LEN]
        else:
            pad_width = MAX_PAD_LEN - melspec_db.shape[1]
            melspec_db = np.pad(melspec_db, pad_width=((0, 0), (0, pad_width)), mode='constant')
            
        return melspec_db
    except Exception as e:
        return None

print(f"\nExtraindo espectrogramas de {len(df_max)} amostras...")
X_mels = []
y_labels = []

for idx, row in tqdm(df_max.iterrows(), total=len(df_max)):
    path_wet = os.path.join(DIR_AUDIO_WET, row['file_name'])
    spec = extrair_espectrograma(path_wet)
    
    if spec is not None:
        X_mels.append(spec)
        y_labels.append(row['keyword'])

# =============================================================================
# 3. PREPARAÇÃO DOS DADOS PARA A CNN
# =============================================================================
# A CNN 2D exige o formato (Lote, Altura, Largura, Canais). 
# Nossos canais = 1 (como se fosse uma imagem em preto e branco)
X = np.array(X_mels)
X = X[..., np.newaxis] 

# Codifica as palavras em números (ex: bright = 0, dark = 1)
le = LabelEncoder()
y = le.fit_transform(y_labels)
num_classes = len(np.unique(y))

# Divisão Treino / Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nShape dos dados de treino: {X_train.shape}") # Ex: (N, 64, 86, 1)

# =============================================================================
# 4. ARQUITETURA DA REDE NEURAL CONVOLUCIONAL (CNN)
# =============================================================================
model = models.Sequential([
    layers.Input(shape=(N_MELS, MAX_PAD_LEN, 1)),
    
    # 1º Bloco Convolucional
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    # 2º Bloco Convolucional
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(0.3), # Evita overfitting
    
    # Achatamento e Classificação
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# =============================================================================
# 5. TREINAMENTO E RESULTADOS
# =============================================================================
print("\nTreinando a CNN...")
# Usamos verbose=1 para você ver a barra de progresso.
history = model.fit(X_train, y_train, epochs=15, batch_size=32, validation_data=(X_test, y_test), verbose=1)

# Avaliação Final
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
acuracia_aleatoria = 1.0 / num_classes

print("\n" + "=" * 80)
print("RESULTADOS DA CNN (ESPECTROGRAMA 2D)")
print("-" * 80)
print(f"  Acurácia do Acaso : {acuracia_aleatoria * 100:.2f}%")
print(f"  Acurácia MLP Antiga: 65.62%")
print(f"  Acurácia CNN Nova : {test_acc * 100:.2f}%")
print("=" * 80)

ModuleNotFoundError: No module named 'tensorflow'

In [6]:
# =============================================================================
# BLOCO 7 — SILHOUETTE SCORE (SEPARAÇÃO GLOBAL DE CLASSES)
# =============================================================================
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

print("\n" + "=" * 80)
print("4. SILHOUETTE SCORE (QUALIDADE DO MAPA SEMÂNTICO)")
print("-" * 80)

# Focamos apenas na intensidade máxima para ver a separação real dos timbres
df_max = df_feat[df_feat['intensity'] == 1.0].copy()

if not df_max.empty and len(df_max['keyword'].unique()) > 1:
    # Extraímos as features e padronizamos
    X_sil = StandardScaler().fit_transform(np.stack(df_max['mfccs'].values))
    y_sil = df_max['keyword'].values
    
    score = silhouette_score(X_sil, y_sil, metric='cosine')
    print(f"  Silhouette Score (MFCCs / Cosine): {score:.4f}")
    
    if score > 0.1:
        print("  ✓ Excelente: Os descritores formam clusters acústicos independentes.")
    elif score > 0.0:
        print("  ! Razoável: Há alguma sobreposição entre descritores similares.")
    else:
        print("  X Alerta: O espaço acústico está muito misturado. O modelo pode se confundir.")
else:
    print("  Dados insuficientes para calcular o Silhouette Score.")


4. SILHOUETTE SCORE (QUALIDADE DO MAPA SEMÂNTICO)
--------------------------------------------------------------------------------
  Silhouette Score (MFCCs / Cosine): -0.0731
  X Alerta: O espaço acústico está muito misturado. O modelo pode se confundir.


In [7]:
# =============================================================================
# COESÃO SEMÂNTICA POR FAMÍLIA (SILHOUETTE SCORE ISOLADO)
# =============================================================================
from sklearn.metrics import silhouette_score

print("\n" + "=" * 80)
print("4.1. SILHOUETTE SCORE ISOLADO POR FAMÍLIA FÍSICA")
print("-" * 80)

# Garante que a coluna 'familia' existe no df_max (usando a mesma lógica do seu bloco 9)
df_max['familia'] = df_max['original_note'].apply(lambda x: x.split('_')[0])

familias_disponiveis = df_max['familia'].unique()

for fam in sorted(familias_disponiveis):
    # Filtra o dataframe para conter apenas os dados da família atual
    df_fam = df_max[df_max['familia'] == fam]
    
    # Precisamos garantir que a família possui mais de um descritor/classe para calcular o score
    if len(df_fam['keyword'].unique()) > 1:
        # Extrai e padroniza as features (MFCCs) apenas para esta família
        X_fam = scaler.fit_transform(np.stack(df_fam['mfccs'].values))
        y_fam = df_fam['keyword'].values
        
        # Calcula o Silhouette Score interno da família usando a métrica cosseno
        score_fam = silhouette_score(X_fam, y_fam, metric='cosine')
        
        # Define um status visual baseado no resultado
        if score_fam > 0.3:
            status = "✓ Excelente Coesão"
        elif score_fam > 0.0:
            status = "◌ Clusters Distinguíveis"
        else:
            status = "X Sobreposto (Ruído)"
            
        print(f"  [{fam.upper():10s}] Silhouette Score: {score_fam:2.4f} -> {status}")
    else:
        print(f"  [{fam.upper():10s}] Ignorado: Classes insuficientes para agrupamento.")
print("=" * 80)


4.1. SILHOUETTE SCORE ISOLADO POR FAMÍLIA FÍSICA
--------------------------------------------------------------------------------
  [BASS      ] Silhouette Score: -0.1170 -> X Sobreposto (Ruído)
  [BRASS     ] Silhouette Score: 0.0309 -> ◌ Clusters Distinguíveis
  [GUITAR    ] Silhouette Score: -0.1397 -> X Sobreposto (Ruído)
  [KEYBOARD  ] Silhouette Score: -0.0970 -> X Sobreposto (Ruído)
  [ORGAN     ] Silhouette Score: 0.0391 -> ◌ Clusters Distinguíveis
  [REED      ] Silhouette Score: -0.1203 -> X Sobreposto (Ruído)
  [STRING    ] Silhouette Score: -0.0434 -> X Sobreposto (Ruído)
  [VOCAL     ] Silhouette Score: 0.0144 -> ◌ Clusters Distinguíveis


In [8]:
# =============================================================================
# BLOCO 8 — CORRELAÇÃO DE PEARSON (LINEARIDADE DO DSP)
# =============================================================================
from scipy.stats import pearsonr

print("\n" + "=" * 80)
print("5. CORRELAÇÃO DE PEARSON (LINEARIDADE DO ESCALONAMENTO)")
print("-" * 80)

for kw, (feat, weight) in feature_alvo.items():
    df_kw = df_feat[df_feat['keyword'] == kw]
    if df_kw.empty: continue
    
    pearsons = []
    for nota in df_kw['original_note'].unique():
        df_n = df_kw[df_kw['original_note'] == nota].sort_values('intensity')
        if len(df_n) < 3: continue
        
        # Pearson avalia a relação linear estrita entre intensidade (0.2 a 1.0) e a feature
        p_coef, _ = pearsonr(df_n['intensity'], df_n[feat])
        if not np.isnan(p_coef): pearsons.append(p_coef * weight)
        
    if pearsons:
        media_pearson = np.mean(pearsons)
        print(f"  [{kw.upper():10s}] Pearson R: {media_pearson:.3f} (Ideal > 0.8)")


5. CORRELAÇÃO DE PEARSON (LINEARIDADE DO ESCALONAMENTO)
--------------------------------------------------------------------------------
  [BRIGHT    ] Pearson R: -0.238 (Ideal > 0.8)
  [DARK      ] Pearson R: 0.253 (Ideal > 0.8)
  [CRISP     ] Pearson R: 0.300 (Ideal > 0.8)
  [MUFFLED   ] Pearson R: 0.770 (Ideal > 0.8)
  [WARM      ] Pearson R: 0.340 (Ideal > 0.8)
  [TINNY     ] Pearson R: -0.371 (Ideal > 0.8)


In [9]:
# =============================================================================
# AVALIAÇÃO DE RIGOR POR FAMÍLIA DE INSTRUMENTO
# =============================================================================
from scipy.stats import kendalltau, pearsonr
import numpy as np

print("=" * 80)
print("CORRELAÇÃO DE PEARSON E KENDALL (ISOLADO POR FAMÍLIA FÍSICA)")
print("=" * 80)

# Cria a coluna de família a partir da string da nota original
# Exemplo: 'guitar_acoustic_015-048-025' -> 'guitar'
df_feat['family'] = df_feat['original_note'].apply(lambda x: x.split('_')[0])

feature_alvo = {
    'bright': ('rolloff', 1),
    'dark':   ('rolloff', -1),
    'crisp':  ('flatness', 1),
    'muffled':('flatness', -1),
    'warm':   ('centroid', -1),
    'tinny':  ('centroid', 1),
}

# Escolha as principais famílias que você usou do NSynth
familias_principais = df_feat['family'].unique()

for familia in familias_principais:
    df_fam = df_feat[df_feat['family'] == familia]
    if len(df_fam) == 0: continue
    
    print(f"\n--- Analisando Família: {familia.upper()} ---")
    
    for kw, (feat, weight) in feature_alvo.items():
        df_kw = df_fam[df_fam['keyword'] == kw]
        if df_kw.empty: continue
        
        taus = []
        pearsons = []
        
        for nota in df_kw['original_note'].unique():
            df_n = df_kw[df_kw['original_note'] == nota].sort_values('intensity')
            if len(df_n) < 3: continue
            
            # Garante que a métrica tem alguma variância para não dar erro matemático
            if np.std(df_n[feat]) > 1e-6:
                # Kendall
                t, _ = kendalltau(df_n['intensity'], df_n[feat])
                if not np.isnan(t): taus.append(t * weight)
                
                # Pearson
                p, _ = pearsonr(df_n['intensity'], df_n[feat])
                if not np.isnan(p): pearsons.append(p * weight)
                
        if taus and pearsons:
            print(f"  [{kw.upper():10s}] | Tau: {np.mean(taus):.3f} | Pearson: {np.mean(pearsons):.3f}")

CORRELAÇÃO DE PEARSON E KENDALL (ISOLADO POR FAMÍLIA FÍSICA)

--- Analisando Família: KEYBOARD ---
  [BRIGHT    ] | Tau: -0.082 | Pearson: -0.040
  [DARK      ] | Tau: 0.191 | Pearson: 0.236
  [CRISP     ] | Tau: 0.348 | Pearson: 0.461
  [MUFFLED   ] | Tau: 0.645 | Pearson: 0.647
  [WARM      ] | Tau: 0.018 | Pearson: -0.029
  [TINNY     ] | Tau: -0.145 | Pearson: -0.029

--- Analisando Família: BASS ---
  [BRIGHT    ] | Tau: 0.077 | Pearson: 0.128
  [DARK      ] | Tau: -0.015 | Pearson: -0.022
  [CRISP     ] | Tau: 0.026 | Pearson: 0.089
  [MUFFLED   ] | Tau: 0.661 | Pearson: 0.759
  [WARM      ] | Tau: -0.133 | Pearson: -0.146
  [TINNY     ] | Tau: 0.092 | Pearson: 0.106

--- Analisando Família: ORGAN ---
  [BRIGHT    ] | Tau: -0.026 | Pearson: -0.080
  [DARK      ] | Tau: 0.035 | Pearson: 0.081
  [CRISP     ] | Tau: 0.147 | Pearson: 0.387
  [MUFFLED   ] | Tau: 0.620 | Pearson: 0.603
  [WARM      ] | Tau: 0.576 | Pearson: 0.589
  [TINNY     ] | Tau: -0.588 | Pearson: -0.627

--- Anal

In [10]:
# =============================================================================
# BLOCO 9 — ROBUSTEZ MULTI-FONTE (INDEPENDÊNCIA INSTRUMENTAL)
# =============================================================================
from scipy.stats import kruskal

print("\n" + "=" * 80)
print("6. ROBUSTEZ DA TRANSFORMAÇÃO ENTRE INSTRUMENTOS (KRUSKAL-WALLIS)")
print("-" * 80)

# Extraímos a família direto do nome da nota contida no df_feat de forma limpa
df_feat['familia'] = df_feat['original_note'].apply(lambda x: x.split('_')[0])

for kw, (feat, weight) in feature_alvo.items():
    # Analisamos apenas a intensidade máxima
    df_teste = df_feat[(df_feat['keyword'] == kw) & (df_feat['intensity'] == 1.0)]
    
    familias = df_teste['familia'].unique()
    if len(familias) > 1:
        # Agrupa os valores da feature alvo (ex: rolloff) separando por família de instrumento
        grupos = [df_teste[df_teste['familia'] == f][feat].values for f in familias]
        
        # O teste verifica se a mediana das distribuições é a mesma entre guitarras, synths, etc.
        stat, p_value = kruskal(*grupos)
        
        # Interpretação: Se p_value for ALTO (> 0.05), significa que a transformação é coesa
        # e o instrumento de origem não "atrapalhou" a criação da feature tímbrica.
        status = "✓ Coeso" if p_value > 0.05 else "! Dependente da Fonte"
        print(f"  [{kw.upper():10s}] p-value: {p_value:.4f} -> {status}")


6. ROBUSTEZ DA TRANSFORMAÇÃO ENTRE INSTRUMENTOS (KRUSKAL-WALLIS)
--------------------------------------------------------------------------------
  [BRIGHT    ] p-value: 0.0000 -> ! Dependente da Fonte
  [DARK      ] p-value: 0.0000 -> ! Dependente da Fonte
  [CRISP     ] p-value: 0.0001 -> ! Dependente da Fonte
  [MUFFLED   ] p-value: 0.0000 -> ! Dependente da Fonte
  [WARM      ] p-value: 0.0000 -> ! Dependente da Fonte
  [TINNY     ] p-value: 0.0000 -> ! Dependente da Fonte


In [11]:
# =============================================================================
# PIPELINE DE AVALIAÇÃO AVANÇADO: ANÁLISE GRANULAR E RELATÓRIO DE EMBEDDINGS
# =============================================================================
import os
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm import tqdm
from scipy.stats import kendalltau, pearsonr, kruskal
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import rbf_kernel

print("=====================================================================")
print("🚀 INICIANDO ANÁLISE ACÚSTICA GRANULAR MULTIVARIÁVEL")
print("=====================================================================")

DIR_META = "dataset_fase2_valid_elite"
CSV_PATH = os.path.join(DIR_META, "metadata.csv")
AUDIO_DIR = os.path.join(DIR_META, "audio")
BLUEPRINT_PATH = os.path.join("dataset_fase1_blueprints_valid", "blueprint_fase1.parquet")

if not os.path.exists(CSV_PATH):
    raise RuntimeError("Arquivo metadata.csv não encontrado!")

df = pd.read_csv(CSV_PATH)

# --- CORREÇÃO DE METADADOS: INJEÇÃO DINÂMICA DA FAMÍLIA ---
print("Restaurando dependências de metadados a partir do Blueprint...")
df_blueprint = pd.read_parquet(BLUEPRINT_PATH)
mapa_familias = dict(zip(df_blueprint['id_par_hard_negative'], df_blueprint['familia_instrumento']))
df['familia_instrumento'] = df['hard_negative_pair_id'].map(mapa_familias)
print(f"✓ {df['familia_instrumento'].notna().sum()} amostras pareadas com suas famílias acústicas.\n")

# --- 1. EXTRAÇÃO DE CARACTERÍSTICAS ACÚSTICAS COMPLEMENTARES ---
features_lista = []
indices_validos = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extraindo Features"):
    audio_path = os.path.join(AUDIO_DIR, row['file_name'])
    if not os.path.exists(audio_path): continue
        
    try:
        y, sr = sf.read(audio_path)
        if len(y.shape) > 1: y = y[:, 0]
            
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfccs_mean = np.mean(mfccs, axis=1)
        
        # Injeção de centroide e rolloff para capturar as caudas de Reverb
        centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        
        feature_vetor = np.concatenate([mfccs_mean, [np.mean(centroid), np.mean(rolloff)]])
        features_lista.append(feature_vetor)
        indices_validos.append(idx)
    except: continue

df_feat = df.loc[indices_validos].copy()
X_raw = np.stack(features_lista)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
df_feat['features_scaled'] = list(X_scaled)

def mmd_rbf(X, Y, gamma=0.1):
    if len(X) == 0 or len(Y) == 0: return 0.0
    XX = rbf_kernel(X, X, gamma)
    YY = rbf_kernel(Y, Y, gamma)
    XY = rbf_kernel(X, Y, gamma)
    return XX.mean() + YY.mean() - 2 * XY.mean()

print(f"\n✓ Matriz de características consolidada: {X_scaled.shape}\n")


# --- METRICA GLOBAL DE CONTROLE ---
print("=" * 90)
score_global = silhouette_score(X_scaled, df_feat['keyword'].values, metric='cosine') if len(np.unique(df_feat['keyword'].values)) > 1 else 0.0
print(f"📊 DIAGNÓSTICO GLOBAL DO ESPAÇO: Silhouette Cosine = {score_global:.4f}")
print("=" * 90)


# =============================================================================
# VISÃO GERAL 1: DETALHAMENTO AGRUPADO POR TIPO DE EFEITO (EQ vs REVERB)
# =============================================================================
print("\n" + "-"*40 + " AGRUPADO POR TIPO DE EFEITO " + "-"*40)
print(f"{'EFEITO':<10} | {'PALAVRA':<12} | {'TAU (MONOT.)':<12} | {'PEARSON R':<10} | {'MMD OPOSTO':<12} | {'SILHOUETTE':<10}")
print("-" * 93)

for dsp_type, sub_df in df_feat.groupby('effect_type'):
    keywords = sorted(sub_df['keyword'].unique())
    
    for kw in keywords:
        df_kw = sub_df[sub_df['keyword'] == kw]
        
        # Monotonicidade e Linearidade
        intensidades = sorted(df_kw['intensity'].unique())
        medias_int = []
        for i in intensidades:
            vetores_int = np.stack(df_kw[df_kw['intensity'] == i]['features_scaled'].values)
            medias_int.append(np.mean(vetores_int))
            
        tau, _ = kendalltau(intensidades, medias_int) if len(intensidades) > 1 else (0, 0)
        r_coeff, _ = pearsonr(intensidades, medias_int) if len(intensidades) > 1 else (0, 0)
        
        # Silhouette Local
        X_kw = np.stack(df_kw['features_scaled'].values)
        labels_kw = df_kw['keyword'].values
        sil = silhouette_score(X_kw, labels_kw, metric='cosine') if len(np.unique(labels_kw)) > 1 else 0.0
        
        # MMD com o Antônimo Geométrico dentro do mesmo efeito
        df_max_kw = df_kw[df_kw['intensity'] == 1.0]
        id_pares = df_max_kw['hard_negative_pair_id'].unique()
        df_oposto = sub_df[(sub_df['hard_negative_pair_id'].isin(id_pares)) & (sub_df['keyword'] != kw) & (sub_df['intensity'] == 1.0)]
        
        mmd = mmd_rbf(np.stack(df_max_kw['features_scaled'].values), np.stack(df_oposto['features_scaled'].values)) if len(df_oposto) > 0 and len(df_max_kw) > 0 else 0.0
        
        print(f"{dsp_type.upper():<10} | {kw:<12} | {tau:>12.3f} | {r_coeff:>10.3f} | {mmd:>12.4f} | {sil:>10.4f}")


# =============================================================================
# VISÃO GERAL 2: DETALHAMENTO AGRUPADO POR FAMÍLIA DE INSTRUMENTO
# =============================================================================
print("\n" + "-"*39 + " AGRUPADO POR FAMÍLIA FÍSICA " + "-"*39)
print(f"{'INSTRUMENTO':<12} | {'SILHOUETTE COESÃO':<20} | {'KRUSKAL P-VALUE (UNIVERSALIDADE DO DSP)':<40}")
print("-" * 93)

for fam, sub_df in df_feat.groupby('familia_instrumento'):
    X_fam = np.stack(sub_df['features_scaled'].values)
    labels_fam = sub_df['keyword'].values
    sil_fam = silhouette_score(X_fam, labels_fam, metric='cosine') if len(np.unique(labels_fam)) > 1 else 0.0
    
    # Kruskal-Wallis por palavra para testar se a fonte altera drasticamente o comportamento físico
    p_values_kw = []
    for kw, df_kw in sub_df.groupby('keyword'):
        df_max = df_kw[df_kw['intensity'] == 1.0]
        if len(df_max) > 1:
            vetores = np.stack(df_max['features_scaled'].values)
            # Como Kruskal espera vetores 1D, tiramos a média das features por amostra
            medias_amostras = np.mean(vetores, axis=1)
            # Para testar dependência da fonte, o Kruskal precisa comparar as distribuições DESTA palavra em VÁRIAS famílias.
            # Essa implementação ajustada isola a métrica.
            pass 
            
    # Reestruturação do Kruskal para comparar entre famílias corretamente
    p_medio = 0.0 
    print(f"{fam.upper():<12} | {sil_fam:>20.4f} | (Processado na Visão Cruzada)")


# =============================================================================
# VISÃO GERAL 3: MATRIZ CRUZADA CRUCIAL (EFEITO × INSTRUMENTO)
# =============================================================================
print("\n" + "-"*36 + " MATRIZ CRUZADA (EFEITO × INSTRUMENTO) " + "-"*36)
print(f"{'INSTRUMENTO':<12} | {'EFEITO':<8} | {'TOP 1 PALAVRA ESTÁVEL':<22} | {'TAU':<7} | {'MMD MÉDIO PARES':<15}")
print("-" * 93)

for (fam, dsp_type), sub_df in df_feat.groupby(['familia_instrumento', 'effect_type']):
    melhor_palavra = "N/A"
    melhor_tau = -2.0
    mmds_do_grupo = []
    
    for kw, df_kw in sub_df.groupby('keyword'):
        intensidades = sorted(df_kw['intensity'].unique())
        medias_int = []
        for i in intensidades:
            vet_int = np.stack(df_kw[df_kw['intensity'] == i]['features_scaled'].values)
            medias_int.append(np.mean(vet_int))
            
        tau, _ = kendalltau(intensidades, medias_int) if len(intensidades) > 1 else (0, 0)
        
        # Ignora NaNs do Kendall se as intensidades gerarem vetor idêntico
        if not np.isnan(tau) and abs(tau) > melhor_tau:
            melhor_tau = abs(tau)
            melhor_palavra = kw
            
        df_max_kw = df_kw[df_kw['intensity'] == 1.0]
        id_pares = df_max_kw['hard_negative_pair_id'].unique()
        df_oposto = sub_df[(sub_df['hard_negative_pair_id'].isin(id_pares)) & (sub_df['keyword'] != kw) & (sub_df['intensity'] == 1.0)]
        
        if len(df_max_kw) > 0 and len(df_oposto) > 0:
            mmd_local = mmd_rbf(np.stack(df_max_kw['features_scaled'].values), np.stack(df_oposto['features_scaled'].values))
            mmds_do_grupo.append(mmd_local)
            
    mmd_medio_grupo = np.mean(mmds_do_grupo) if mmds_do_grupo else 0.0
    
    print(f"{fam.upper():<12} | {dsp_type.upper():<8} | {melhor_palavra:<22} | {melhor_tau:>7.3f} | {mmd_medio_grupo:>15.4f}")

# =============================================================================
# AVALIAÇÃO DE DEPENDÊNCIA DA FONTE (KRUSKAL GLOBAL)
# =============================================================================
print("\n" + "-"*35 + " KRUSKAL-WALLIS: DEPENDÊNCIA DA FONTE " + "-"*35)
keywords_globais = df_feat['keyword'].unique()

for kw in keywords_globais:
    df_kw = df_feat[(df_feat['keyword'] == kw) & (df_feat['intensity'] == 1.0)]
    grupos_por_familia = []
    
    for fam, group in df_kw.groupby('familia_instrumento'):
        if len(group) > 1:
            vetores = np.stack(group['features_scaled'].values)
            grupos_por_familia.append(np.mean(vetores, axis=1))
            
    if len(grupos_por_familia) > 1:
        stat, p_value = kruskal(*grupos_por_familia)
        status = "! Dependente (Source-Dependent)" if p_value < 0.05 else "✓ Universal"
        print(f"  [{kw:<12}] p-value: {p_value:.4f} -> {status}")

print("\n" + "=" * 90)
print("✓ ANÁLISE COMPACTA E MULTIVARIÁVEL EXECUTADA!")
print("=" * 90)

🚀 INICIANDO ANÁLISE ACÚSTICA GRANULAR MULTIVARIÁVEL
Restaurando dependências de metadados a partir do Blueprint...
✓ 9600 amostras pareadas com suas famílias acústicas.



Extraindo Features: 100%|██████████████████████████████████████████████████████████| 9600/9600 [04:40<00:00, 34.23it/s]



✓ Matriz de características consolidada: (9600, 15)

📊 DIAGNÓSTICO GLOBAL DO ESPAÇO: Silhouette Cosine = -0.0907

---------------------------------------- AGRUPADO POR TIPO DE EFEITO ----------------------------------------
EFEITO     | PALAVRA      | TAU (MONOT.) | PEARSON R  | MMD OPOSTO   | SILHOUETTE
---------------------------------------------------------------------------------------------
EQ         | bright       |        1.000 |      0.981 |       0.5281 |     0.0000
EQ         | crisp        |       -1.000 |     -0.989 |       0.3056 |     0.0000
EQ         | dark         |       -1.000 |     -0.992 |       0.5281 |     0.0000
EQ         | harsh        |        1.000 |      0.994 |       0.2742 |     0.0000
EQ         | mellow       |        1.000 |      0.987 |       0.0136 |     0.0000
EQ         | metallic     |        1.000 |      0.993 |       0.0136 |     0.0000
EQ         | muffled      |        1.000 |      0.980 |       0.3056 |     0.0000
EQ         | soft        